# Task 1 — Coarse-Grained Animal Classification
COMP30027 Machine Learning 2026 — Project 2

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

# Display settings
pd.set_option('display.max_columns', 20)
%matplotlib inline

## 2. Load Data

In [ ]:
DATA_DIR = Path('task1_data')

# Metadata
train_meta = pd.read_csv(DATA_DIR / 'train_metadata.csv')
test_meta  = pd.read_csv(DATA_DIR / 'test_metadata.csv')

# Feature CSVs — index by image_id so we can look up rows by ID later
color_hist = pd.read_csv(DATA_DIR / 'color_histogram.csv',     index_col='image_id')
hog_pca    = pd.read_csv(DATA_DIR / 'hog_pca.csv',             index_col='image_id')
add_feat   = pd.read_csv(DATA_DIR / 'additional_features.csv', index_col='image_id')

print(f'Train samples : {len(train_meta)}')
print(f'Test  samples : {len(test_meta)}')
print(f'color_histogram   : {color_hist.shape}')
print(f'hog_pca           : {hog_pca.shape}')
print(f'additional_features: {add_feat.shape}')

In [ ]:
# Preview metadata
train_meta.head()

In [ ]:
test_meta.head()

## 3. Data Quality Checks

In [ ]:
# Check for missing values across all dataframes
for name, df in [('train_meta', train_meta), ('test_meta', test_meta),
                 ('color_hist', color_hist), ('hog_pca', hog_pca), ('add_feat', add_feat)]:
    nulls = df.isnull().sum().sum()
    print(f'{name:20s}  missing values: {nulls}')

In [ ]:
# Confirm feature CSVs cover all train + test images (should be 3750 + 1250 = 5000 rows)
all_ids    = list(train_meta['image_id']) + list(test_meta['image_id'])
in_color   = set(color_hist.index)
in_hog     = set(hog_pca.index)
in_add     = set(add_feat.index)

missing_color = [i for i in all_ids if i not in in_color]
missing_hog   = [i for i in all_ids if i not in in_hog]
missing_add   = [i for i in all_ids if i not in in_add]

print(f'IDs missing from color_histogram   : {len(missing_color)}')
print(f'IDs missing from hog_pca           : {len(missing_hog)}')
print(f'IDs missing from additional_features: {len(missing_add)}')

## 4. Explore the Training Labels

In [ ]:
# Class distribution
class_counts = train_meta['class_name'].value_counts().sort_index()
print(class_counts)
print(f'\nTotal classes : {train_meta["class_name"].nunique()}')
print(f'Min per class : {class_counts.min()}')
print(f'Max per class : {class_counts.max()}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(class_counts.index, class_counts.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Class')
ax.set_ylabel('Number of images')
ax.set_title('Task 1 — Training set class distribution')
ax.axhline(class_counts.mean(), color='tomato', linestyle='--', label=f'Mean = {class_counts.mean():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Sample Images per Class

In [ ]:
classes = sorted(train_meta['class_name'].unique())
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, cls in zip(axes.flat, classes):
    sample = train_meta[train_meta['class_name'] == cls].iloc[0]
    img = Image.open(DATA_DIR / sample['image_path'])
    ax.imshow(img)
    ax.set_title(cls, fontsize=11)
    ax.axis('off')

plt.suptitle('Task 1 — One sample image per class', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Explore the Provided Features

In [ ]:
# Summary statistics for additional_features (most interpretable)
add_feat.describe().round(3)

In [ ]:
# Distribution of first 5 columns from each feature set
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train_ids = train_meta['image_id'].values
add_feat.loc[train_ids].iloc[:, :5].plot(kind='hist', bins=30, alpha=0.6, ax=axes[0])
axes[0].set_title('additional_features (first 5 cols)')
axes[0].set_xlabel('Value')

hog_pca.loc[train_ids].iloc[:, :5].plot(kind='hist', bins=30, alpha=0.6, ax=axes[1])
axes[1].set_title('hog_pca (first 5 PCs)')
axes[1].set_xlabel('Value')

color_hist.loc[train_ids].iloc[:, :5].plot(kind='hist', bins=30, alpha=0.6, ax=axes[2])
axes[2].set_title('color_histogram (first 5 bins)')
axes[2].set_xlabel('Value')

plt.tight_layout()
plt.show()

In [ ]:
# Check feature variance — low-variance columns may not be useful
add_var = add_feat.loc[train_ids].var().sort_values()
print('additional_features — lowest variance columns:')
print(add_var.head(5).round(6))
print('\nadditional_features — highest variance columns:')
print(add_var.tail(5).round(6))

## 7. Feature Engineering

### 7.1 feat_basic — Combine and scale provided features

Concatenate the 3 provided CSVs into one matrix, then apply `StandardScaler`. Scaling is required because kNN and SVM are distance-based — without it, features with large numeric ranges (e.g. `feat_1` variance ~4.5M seen above) would dominate distance calculations and bias the model.

In [ ]:
from sklearn.preprocessing import StandardScaler

train_ids = train_meta['image_id'].values
test_ids  = test_meta['image_id'].values

# Concatenate the 3 feature CSVs, aligned by image_id
X_train_raw = pd.concat([
    color_hist.loc[train_ids],
    hog_pca.loc[train_ids],
    add_feat.loc[train_ids],
], axis=1).values

X_test_raw = pd.concat([
    color_hist.loc[test_ids],
    hog_pca.loc[test_ids],
    add_feat.loc[test_ids],
], axis=1).values

y_train = train_meta['class_id'].values

# Fit scaler on train only, then apply to test
scaler_basic  = StandardScaler()
X_train_basic = scaler_basic.fit_transform(X_train_raw)
X_test_basic  = scaler_basic.transform(X_test_raw)

print(f'feat_basic — train : {X_train_basic.shape}  (3750 images × {X_train_basic.shape[1]} features)')
print(f'feat_basic — test  : {X_test_basic.shape}')

### 7.2 feat_combined — Engineer additional features from raw pixels

We extract the mean and standard deviation for each RGB and HSV channel from every raw image (12 values per image). This is cheap to compute but captures information the provided features don't: absolute colour tone (hue, saturation) and per-channel brightness spread — useful for visually distinct classes like grey elephants, green frogs, and colourful butterflies.

In [ ]:
import matplotlib.colors as mcolors

def extract_pixel_stats(meta_df, data_dir):
    rows = []
    for _, row in meta_df.iterrows():
        img = Image.open(data_dir / row['image_path']).convert('RGB')
        img_rgb = np.array(img, dtype=np.float32) / 255.0  # (H, W, 3) in [0, 1]
        img_hsv = mcolors.rgb_to_hsv(img_rgb)              # (H, W, 3) in [0, 1]
        stats = np.concatenate([
            img_rgb.mean(axis=(0, 1)),  # R, G, B mean
            img_rgb.std(axis=(0, 1)),   # R, G, B std
            img_hsv.mean(axis=(0, 1)),  # H, S, V mean
            img_hsv.std(axis=(0, 1)),   # H, S, V std
        ])
        rows.append(stats)
    return np.array(rows)

print('Extracting pixel stats for train...')
train_pixel_stats = extract_pixel_stats(train_meta, DATA_DIR)
print('Extracting pixel stats for test...')
test_pixel_stats  = extract_pixel_stats(test_meta,  DATA_DIR)
print(f'Done — shape: {train_pixel_stats.shape}  (12 features per image)')

In [ ]:
# Scale pixel stats (fit on train only), then concatenate with feat_basic
scaler_pixel       = StandardScaler()
train_pixel_scaled = scaler_pixel.fit_transform(train_pixel_stats)
test_pixel_scaled  = scaler_pixel.transform(test_pixel_stats)

X_train_combined = np.concatenate([X_train_basic, train_pixel_scaled], axis=1)
X_test_combined  = np.concatenate([X_test_basic,  test_pixel_scaled],  axis=1)

print(f'feat_combined — train : {X_train_combined.shape}  ({X_train_basic.shape[1]} provided + 12 engineered)')
print(f'feat_combined — test  : {X_test_combined.shape}')

### 7.3 PCA visualisation — Do classes separate in feature space?

Reduce `feat_combined` to 2 dimensions to get a visual sense of class separability before training any model. Well-separated clusters suggest the features carry enough signal for classification.

In [ ]:
from sklearn.decomposition import PCA

class_names = train_meta['class_name'].values  # array of class name per training image

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_train_combined)

unique_classes = sorted(np.unique(class_names))
colours = plt.cm.tab10(np.linspace(0, 1, len(unique_classes)))

fig, ax = plt.subplots(figsize=(9, 6))
for cls, c in zip(unique_classes, colours):
    mask = class_names == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=cls, alpha=0.4, s=15, color=c)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance explained)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance explained)')
ax.set_title('PCA of feat_combined — Task 1 training set')
ax.legend(fontsize=8, markerscale=2, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

print(f'Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 8. Models

We evaluate all models using **5-fold stratified cross-validation** on the training set. Stratified means each fold preserves the class ratio — important for fair evaluation. We report mean accuracy ± standard deviation across folds.

We never touch the test set here — it is only used in Section 9 for the final Kaggle submission.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []  # collect all results for the summary table at the end

def evaluate(name, model, X, y):
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    mean, std = scores.mean(), scores.std()
    print(f'{name:45s}  acc = {mean:.4f}  ±{std:.4f}')
    results.append({'Model': name, 'Mean Acc': mean, 'Std': std})
    return mean

### 8.1 Model 1 — ZeroR Baseline

Always predicts the majority class regardless of input. This is the absolute floor — any model below this accuracy is worse than doing nothing. With 10 balanced classes of 375 images each, ZeroR should score exactly 10%.

In [ ]:
evaluate('ZeroR (majority class)', DummyClassifier(strategy='most_frequent'), X_train_basic, y_train)

### 8.2 Model 2 — k-Nearest Neighbours

kNN classifies a point by majority vote among its k nearest neighbours in feature space. We try several values of k to find the best — small k overfits (high variance), large k underfits (high bias). We test both feature sets because kNN is sensitive to the number and scale of features.

In [ ]:
for k in [1, 5, 11, 21]:
    evaluate(f'kNN  k={k:<3d}  feat_basic',    KNeighborsClassifier(n_neighbors=k), X_train_basic,    y_train)
    evaluate(f'kNN  k={k:<3d}  feat_combined', KNeighborsClassifier(n_neighbors=k), X_train_combined, y_train)

### 8.3 Model 3 — SVM with RBF Kernel

SVM finds the maximum-margin hyperplane separating classes. The RBF kernel maps data into a higher-dimensional space where non-linearly separable classes can be split — well suited to image features where class boundaries are curved. `C` controls the margin/error trade-off; `gamma='scale'` sets the kernel width automatically relative to the number of features.

In [ ]:
evaluate('SVM RBF  C=1   feat_basic',    SVC(kernel='rbf', C=1,  gamma='scale', random_state=42), X_train_basic,    y_train)
evaluate('SVM RBF  C=1   feat_combined', SVC(kernel='rbf', C=1,  gamma='scale', random_state=42), X_train_combined, y_train)
evaluate('SVM RBF  C=10  feat_basic',    SVC(kernel='rbf', C=10, gamma='scale', random_state=42), X_train_basic,    y_train)
evaluate('SVM RBF  C=10  feat_combined', SVC(kernel='rbf', C=10, gamma='scale', random_state=42), X_train_combined, y_train)

### 8.4 Model 4 — SVM with Linear Kernel

The linear kernel finds a straight hyperplane boundary. Counts as a **distinct model** from the RBF SVM per the spec — different kernel = fundamentally different decision boundary shape. Useful for comparison: if linear ≈ RBF, the classes are linearly separable; if RBF >> linear, the boundaries are curved and the kernel trick is doing real work.

In [ ]:
evaluate('SVM Linear  C=1   feat_basic',    SVC(kernel='linear', C=1, random_state=42), X_train_basic,    y_train)
evaluate('SVM Linear  C=1   feat_combined', SVC(kernel='linear', C=1, random_state=42), X_train_combined, y_train)

### 8.5 Results Summary Table

In [ ]:
results_df = pd.DataFrame(results).sort_values('Mean Acc', ascending=False).reset_index(drop=True)
results_df['Mean Acc'] = results_df['Mean Acc'].map('{:.4f}'.format)
results_df['Std']      = results_df['Std'].map('{:.4f}'.format)
results_df.index += 1  # rank starts at 1
results_df

## 9. Error Analysis

Best model: **SVM RBF C=10, feat_combined**. We use `cross_val_predict` to obtain out-of-fold predictions for every training image — each image is predicted by a model that was never trained on it, giving a realistic picture of errors without using the test set.

### 9.1 Confusion Matrix

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

best_model = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
y_pred_cv  = cross_val_predict(best_model, X_train_combined, y_train, cv=cv)

# Class labels in class_id order (0–9)
class_labels = (train_meta.drop_duplicates('class_id')
                           .sort_values('class_id')['class_name']
                           .tolist())

cm = confusion_matrix(y_train, y_pred_cv)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_xlabel('Predicted label', fontsize=11)
ax.set_ylabel('True label', fontsize=11)
ax.set_title('Confusion matrix — SVM RBF C=10, feat_combined (5-fold CV)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 9.2 Per-Class Accuracy & Most Confused Pairs

In [ ]:
# Per-class accuracy (diagonal / row sum)
per_class_acc = cm.diagonal() / cm.sum(axis=1)
acc_df = pd.DataFrame({'Class': class_labels, 'Accuracy': per_class_acc}).sort_values('Accuracy')
print('Per-class accuracy (worst → best):')
print(acc_df.to_string(index=False))

# Top confused pairs (off-diagonal with highest counts)
print('\nTop 5 most confused pairs:')
rows, cols = np.where(np.eye(len(class_labels)) == 0)
pairs = [(cm[r, c], class_labels[r], class_labels[c]) for r, c in zip(rows, cols)]
pairs.sort(reverse=True)
for count, true_cls, pred_cls in pairs[:5]:
    print(f'  True: {true_cls:12s}  →  Predicted: {pred_cls:12s}  ({count} times)')

### 9.3 Misclassified Images

Displaying misclassified examples side by side — the true label is in the title (top) and the predicted label below it. This helps identify whether errors are caused by genuinely ambiguous images or systematic feature failures.

In [ ]:
id_to_name = dict(zip(train_meta['class_id'], train_meta['class_name']))
misclassified_idx = np.where(y_pred_cv != y_train)[0]
print(f'Total misclassified: {len(misclassified_idx)} / {len(y_train)}  ({len(misclassified_idx)/len(y_train)*100:.1f}%)')

fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for ax, idx in zip(axes.flat, misclassified_idx[:18]):
    row = train_meta.iloc[idx]
    img = Image.open(DATA_DIR / row['image_path'])
    ax.imshow(img)
    true_cls = id_to_name[y_train[idx]]
    pred_cls = id_to_name[y_pred_cv[idx]]
    ax.set_title(f'True: {true_cls}\nPred: {pred_cls}', fontsize=7,
                 color='red' if true_cls != pred_cls else 'green')
    ax.axis('off')

plt.suptitle('Misclassified examples — SVM RBF C=10, feat_combined', fontsize=12)
plt.tight_layout()
plt.show()

## 10. Kaggle Submission

Fit the best model on **all** 3750 training images (not just a fold), predict the 1250 test images, and save the CSV.

In [ ]:
best_model.fit(X_train_combined, y_train)
y_test_pred = best_model.predict(X_test_combined)

# Kaggle expects: image_id, class_id (numeric)
submission = pd.DataFrame({
    'image_id': test_meta['image_id'],
    'class_id': y_test_pred
})
submission.to_csv('task1_submission.csv', index=False)
print(f'Saved task1_submission.csv  ({len(submission)} rows)')
submission.head()

## 11. Improved Feature Extraction — ResNet-50

The provided features (HOG, colour histogram, pixel stats) are hand-crafted and shallow. A ResNet-50 pre-trained on ImageNet extracts deep semantic features — edges, textures, object parts — learned from 1.2 million images. We remove its final classification layer and use the 2048-dimensional penultimate layer as a feature vector for each image.

This is explicitly allowed by the spec: *"You may use models pre-trained on other datasets (e.g., an ImageNet-trained ResNet) as generic feature extractors, provided the model was not specifically trained on either of the project datasets."*

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as T

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# Load ResNet-50 with ImageNet weights, strip the classifier head
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = torch.nn.Identity()  # output is now 2048-dim
resnet = resnet.to(device).eval()

# Standard ImageNet normalisation
transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

def extract_resnet(meta_df, data_dir, batch_size=64):
    all_feats = []
    paths = [str(data_dir / p) for p in meta_df['image_path']]
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        imgs  = torch.stack([transform(Image.open(p).convert('RGB')) for p in batch]).to(device)
        with torch.no_grad():
            feats = resnet(imgs).cpu().numpy()
        all_feats.append(feats)
        if i % 320 == 0:
            print(f'  {i+len(batch)}/{len(paths)}')
    return np.concatenate(all_feats, axis=0)

print('Extracting ResNet-50 features — train...')
X_train_resnet = extract_resnet(train_meta, DATA_DIR)
print('Extracting ResNet-50 features — test...')
X_test_resnet  = extract_resnet(test_meta,  DATA_DIR)
print(f'ResNet feature shape: {X_train_resnet.shape}')

In [ ]:
# Scale ResNet features and build feat_resnet_combined
scaler_resnet      = StandardScaler()
X_train_resnet_sc  = scaler_resnet.fit_transform(X_train_resnet)
X_test_resnet_sc   = scaler_resnet.transform(X_test_resnet)

# Also try stacking ResNet on top of feat_combined
X_train_resnet_all = np.concatenate([X_train_combined, X_train_resnet_sc], axis=1)
X_test_resnet_all  = np.concatenate([X_test_combined,  X_test_resnet_sc],  axis=1)

print(f'feat_resnet          : {X_train_resnet_sc.shape}')
print(f'feat_resnet_all      : {X_train_resnet_all.shape}  (resnet + combined)')

### 11.1 Models on ResNet features

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# SVM RBF on ResNet only
evaluate('SVM RBF  C=10  feat_resnet',     SVC(kernel='rbf', C=10, gamma='scale', random_state=42), X_train_resnet_sc,  y_train)
evaluate('SVM RBF  C=10  feat_resnet_all', SVC(kernel='rbf', C=10, gamma='scale', random_state=42), X_train_resnet_all, y_train)

# Logistic Regression — fast and strong on high-dim features
evaluate('LogReg  feat_resnet',            LogisticRegression(max_iter=1000, random_state=42),       X_train_resnet_sc,  y_train)
evaluate('LogReg  feat_resnet_all',        LogisticRegression(max_iter=1000, random_state=42),       X_train_resnet_all, y_train)

# Random Forest
evaluate('RF 300  feat_resnet',            RandomForestClassifier(n_estimators=300, random_state=42), X_train_resnet_sc, y_train)

### 11.2 Updated Kaggle Submission — best ResNet model

In [ ]:
# Update X_best / model_best after reviewing results above
model_best = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
X_best_train = X_train_resnet_sc   # update if feat_resnet_all scores higher
X_best_test  = X_test_resnet_sc

model_best.fit(X_best_train, y_train)
y_test_pred = model_best.predict(X_best_test)

submission = pd.DataFrame({
    'image_id': test_meta['image_id'],
    'class_id': y_test_pred
})
submission.to_csv('task1_submission.csv', index=False)
print(f'Saved task1_submission.csv  ({len(submission)} rows)')
submission['class_id'].value_counts().sort_index()